In [ ]:
import os
import urllib
import urllib.request
import json


class CryptoClass:
    def __init__(self, path_to_crypto):
        # Check if file exists ??? hmm?
        assert os.path.isfile(path_to_crypto), "No file brooooooo"
        
        # Get the file path
        self.path_to_crypto = path_to_crypto
        
        # API link
        self.api_link = "https://api.binance.com/api/v3/ticker/price?symbol="
        
        # Dict to store coins
        self.all_coins = {}
        
        # Read the file and load coins
        self._load_coins()
    
    def _load_coins(self):
        
        try:
            # Open file
            with open(self.path_to_crypto, 'r') as f:
                for line in f:
                    line = line.strip()  # Remove whitespace
                    if line:  
                        parts = line.split()  # Split by TAB
                        print(parts)
                        assert len(parts) == 2, "Invalid format in file na kub"
                        
                        coin_symbol = parts[0].upper()  # Get coin name
                        coin_amount = float(parts[1])   # Get amount
                        self.all_coins[coin_symbol] = coin_amount
        except Exception as e:
            print("Error: " + str(e))
            exit(1)
    
    def get_all_coin_symbols(self):
        
        return self.all_coins.keys()
    
    def get_exchange_rate(self, coin_symbol : str):
        
        try:
            # Create the API URL
            
            url = self.api_link + coin_symbol.upper() + "USDT"
            
            # Open URL
            
            with urllib.request.urlopen(url) as response:
                raw_coin_data = response.read().decode('utf-8')
            
            
            # Find the price value
            coin_data = json.loads(raw_coin_data)
            price_str = coin_data['price']
            
            # Convert to float
            price = float(price_str)
            return price
            
        except Exception as e:
            print("Error getting exchange rate for " + coin_symbol + ": " + str(e))
            return 0.0
    
    def get_net_worth(self):
        
        total = 0.0
        
        for coin_symbol, amount in self.all_coins.items():
            rate = self.get_exchange_rate(coin_symbol)
            total += amount * rate
        
        return total
    
    def buy_coin(self, coin, amount : int):
        
        try:
            assert amount > 0, "Amount must be positive"
            
            coin = coin.upper()
            
            
            if coin in self.all_coins:
                self.all_coins[coin] += amount
            else:
                self.all_coins[coin] = amount
            
            # Save to file
            self._save_coins()
            
            print("Successfully bought " + str(amount) + " " + coin)
            
        except Exception as e:
            print("Error buying coin: " + str(e))
    
    def sell_coin(self, coin, amount : int):
        
        try:
            assert amount > 0, "Amount must be positive"
            
            coin = coin.upper()
            
            # Check if have this coin
            assert coin in self.all_coins, "You don't have this coin"
            
            # Check if have enough
            assert self.all_coins[coin] >= amount, "Not enough coins to sell"
            
            
            self.all_coins[coin] -= amount
            
            
            
            # Save to file
            self._save_coins()
            
            print("Successfully sold " + str(amount) + " " + coin)
            
        except Exception as e:
            print("Error selling coin: " + str(e))
    
    def _save_coins(self):
        
        try:
            
            with open(self.path_to_crypto, 'w') as f:
                for coin_symbol, amount in self.all_coins.items():
                    
                    f.write(coin_symbol + '\t' + str(amount) + '\n')
        except Exception as e:
            print("Error saving to file: " + str(e))
            raise

In [ ]:
plu = CryptoClass('my_crypto.txt')

In [ ]:
plu.sell_coin('ADA',100)

In [ ]:
plu.buy_coin('ADA',100)

In [ ]:
plu.get_exchange_rate('BTC')

In [ ]:
plu.get_net_worth()